To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://github.com/unslothai/unsloth?tab=readme-ov-file#-installation-instructions).

This version of the notebook is adapted for our 100k balanced LunarLander dataset and Qwen model:
- Model: `Qwen/Qwen2.5-0.5B`
- Dataset: `Ali2023kosemen/lunarlander_3_layer` (100k balanced)
- Task: supervised fine-tuning for LunarLander state -> action predictions

The dataset rows use a `conversations` field in ShareGPT style, so we will map those messages into a Qwen-compatible chat template before training.


## 18_03 100k Dataset Run

Bu notebook, `Ali2023kosemen/lunarlander_3_layer` veri seti ile 100k balanced fine-tuning denemesi icin hazirlandi.

Dataset linki: https://huggingface.co/datasets/Ali2023kosemen/lunarlander_3_layer


In [1]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

* We support Llama, Mistral, Phi-3, Gemma, Yi, DeepSeek, Qwen, TinyLlama, Vicuna, Open Hermes etc
* We support 16bit LoRA or 4bit QLoRA. Both 2x faster.
* `max_seq_length` can be set to anything, since we do automatic RoPE Scaling via [kaiokendev's](https://kaiokendev.github.io/til) method.
* [**NEW**] We make Gemma-2 9b / 27b **2x faster**! See our [Gemma-2 9b notebook](https://colab.research.google.com/drive/1vIrqH5uYDQwsJ4-OO3DErvuv4pBgVwk4?usp=sharing)
* [**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)
* [**NEW**] We make Mistral NeMo 12B 2x faster and fit in under 12GB of VRAM! [Mistral NeMo notebook](https://colab.research.google.com/drive/17d3U-CAIwzmbDRqbZ9NnpHxCkmXB6LZ0?usp=sharing)

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Our LunarLander prompts are short, so 1024 is enough.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Great for Colab.

# 4bit models we can use. For this project we will fine-tune Qwen2.5-0.5B.
fourbit_models = [
    "unsloth/Qwen2.5-0.5B-bnb-4bit",
    "unsloth/Qwen2.5-1.5B-bnb-4bit",
    "unsloth/Qwen2.5-3B-bnb-4bit",
    "unsloth/Qwen2.5-7B-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/521M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


<a name="Data"></a>
### Data Prep
We now use our 100k balanced LunarLander dataset from Hugging Face Hub: [`Ali2023kosemen/lunarlander_3_layer`](https://huggingface.co/datasets/Ali2023kosemen/lunarlander_3_layer).

This is not an Alpaca instruction dataset. Our dataset already stores LunarLander examples in a ShareGPT-style conversational format under the `conversations` field. Each sample looks roughly like this:

```json
{
  "conversations": [
    {"from": "human", "value": "State: [...]. What action should the lander take?"},
    {"from": "gpt", "value": "Action: 0 (do nothing)."}
  ]
}
```

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** We still add the **EOS_TOKEN** to the formatted output, otherwise generations can run on too long.

Since Qwen works well with ChatML-style conversations, we will map our `human/gpt` dataset into a Qwen-compatible chat template.


In [4]:
EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
)

def formatting_prompts_func(examples):
    conversations = examples["conversations"]
    texts = []
    for convo in conversations:
        text = tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

from datasets import load_dataset
dataset = load_dataset("Ali2023kosemen/lunarlander_3_layer_3_layer", split = "train")
print(dataset[0])
dataset = dataset.map(formatting_prompts_func, batched = True,)
print(dataset[0]["text"][:1000])


Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


README.md:   0%|          | 0.00/355 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

{'conversations': [{'from': 'human', 'value': 'State: [x=0.0887, y=-0.0011, vx=-0.0002, vy=-0.0000, angle=-0.0027, angular_vel=0.0000, left_leg=1.0000, right_leg=0.0000]. What action should the lander take?'}, {'from': 'gpt', 'value': 'Action: 0 (do nothing).'}]}


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

<|im_start|>user
State: [x=0.0887, y=-0.0011, vx=-0.0002, vy=-0.0000, angle=-0.0027, angular_vel=0.0000, left_leg=1.0000, right_leg=0.0000]. What action should the lander take?<|im_end|>
<|im_start|>assistant
Action: 0 (do nothing).<|im_end|>
<|endoftext|>


<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer).

We are doing supervised fine-tuning here because the dataset already contains target answers for each LunarLander state prompt.
If you want a longer run, increase `max_steps` or switch to `num_train_epochs=1`.


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [6]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
0.566 GB of memory reserved.


In [7]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,3.002942
2,3.012417
3,2.909940
4,2.706568
5,2.411764
6,2.197814
7,1.894589
8,1.668555
9,1.449527
10,1.279833


In [8]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

75.6407 seconds used for training.
1.26 minutes used for training.
Peak reserved memory = 0.914 GB.
Peak reserved memory for training = 0.348 GB.
Peak reserved memory % of max memory = 6.276 %.
Peak reserved memory for training % of max memory = 2.39 %.


<a name="Inference"></a>
### Inference
Let's run the model! We now test it with a LunarLander state prompt in the same style as the training set.

This is useful for checking whether the fine-tuned model learned the expected action output format.


In [9]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

inputs = tokenizer.apply_chat_template(
    [
        {"from": "human", "value": "State: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?"},
    ],
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)


['<|im_start|>user\nState: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?<|im_end|>\n<|im_start|>assistant\nAction: 1 (fire left engine).<|im_end|>']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [10]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

from transformers.utils import logging as hf_logging
from transformers import TextStreamer
hf_logging.set_verbosity_error()

inputs = tokenizer.apply_chat_template(
    [
        {"from": "human", "value": "State: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?"},
    ],
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)


<|im_start|>user
State: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?<|im_end|>
<|im_start|>assistant
Action: 1 (fire left engine).<|im_end|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [11]:
SAVE_DIR = "qwen2.5-0.5b-lunarlander-action-100k-lora"
HF_LORA_REPO_ID = "alirizaercan/qwen2.5-0.5b-lunarlander-action-100k-lora"
HF_LORA_REPO_URL = "https://huggingface.co/alirizaercan/qwen2.5-0.5b-lunarlander-action-100k-lora"
HF_GGUF_REPO_ID = "alirizaercan/qwen2.5-0.5b-lunarlander-action-100k-gguf"
HF_GGUF_REPO_URL = "https://huggingface.co/alirizaercan/qwen2.5-0.5b-lunarlander-action-100k-gguf"

model.save_pretrained(SAVE_DIR) # Local saving
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved locally to: {SAVE_DIR}")
print(f"LoRA repo: {HF_LORA_REPO_URL}")
print(f"GGUF repo: {HF_GGUF_REPO_URL}")
# model.push_to_hub(HF_LORA_REPO_ID, token = "...") # Online saving
# tokenizer.push_to_hub(HF_LORA_REPO_ID, token = "...") # Online saving


Saved locally to: qwen2.5-0.5b-lunarlander-action-lora
LoRA repo: https://huggingface.co/alirizaercan/qwen2.5-0.5b-lunarlander-action-lora
GGUF repo: https://huggingface.co/alirizaercan/qwen2.5-0.5b-lunarlander-action-gguf


### Download the saved LoRA adapter
After saving the adapter locally in Colab, we zip the folder and download it to our computer.

This is useful before uploading the files to Hugging Face Hub.


In [13]:
!zip -r qwen2.5-0.5b-lunarlander-action-100k-lora.zip qwen2.5-0.5b-lunarlander-action-100k-lora

from google.colab import files
files.download("qwen2.5-0.5b-lunarlander-action-100k-lora.zip")


updating: qwen2.5-0.5b-lunarlander-action-lora/ (stored 0%)
updating: qwen2.5-0.5b-lunarlander-action-lora/adapter_model.safetensors (deflated 7%)
updating: qwen2.5-0.5b-lunarlander-action-lora/chat_template.jinja (deflated 75%)
updating: qwen2.5-0.5b-lunarlander-action-lora/adapter_config.json (deflated 58%)
updating: qwen2.5-0.5b-lunarlander-action-lora/tokenizer.json (deflated 81%)
updating: qwen2.5-0.5b-lunarlander-action-lora/tokenizer_config.json (deflated 38%)
updating: qwen2.5-0.5b-lunarlander-action-lora/README.md (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [14]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

from transformers.utils import logging as hf_logging
from transformers import TextStreamer
hf_logging.set_verbosity_error()

inputs = tokenizer.apply_chat_template(
    [
        {"from": "human", "value": "State: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?"},
    ],
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)


<|im_start|>user
State: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?<|im_end|>
<|im_start|>assistant
Action: 1 (fire left engine).<|im_end|>


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [15]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR LUNARLANDER MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [16]:
HF_LORA_REPO_ID = "alirizaercan/qwen2.5-0.5b-lunarlander-action-100k-lora"

# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged(HF_LORA_REPO_ID, tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged(HF_LORA_REPO_ID, tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged(HF_LORA_REPO_ID, tokenizer, save_method = "lora", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [17]:
HF_GGUF_REPO_ID = "alirizaercan/qwen2.5-0.5b-lunarlander-action-100k-gguf"

# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
if False: model.push_to_hub_gguf(HF_GGUF_REPO_ID, tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf(HF_GGUF_REPO_ID, tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
# if False: model.push_to_hub_gguf(HF_GGUF_REPO_ID, tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        HF_GGUF_REPO_ID,
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m"],
        token = "", # Get a token at https://huggingface.co/settings/tokens
)


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/774 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:15<00:00, 15.70s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:05<00:00,  6.00s/it]


Unsloth: Merge process complete. Saved to `/content/model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf/qwen2.5-0.5b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf/qwen2.5-0.5b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/qwen2.5-0.5b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model model_gguf/qwen2.5-0.5b.Q4_K_M.gguf -p "why is the sky blue?"


In [18]:
!ls -lh model

total 954M
-rw-r--r-- 1 root root  836 Mar 17 11:57 chat_template.jinja
-rw-r--r-- 1 root root 1.5K Mar 17 11:57 config.json
-rw-r--r-- 1 root root 943M Mar 17 11:57 model.safetensors
-rw-r--r-- 1 root root 1.2K Mar 17 11:57 tokenizer_config.json
-rw-r--r-- 1 root root  11M Mar 17 11:57 tokenizer.json


In [19]:
!zip -r qwen2.5-0.5b-lunarlander-action-100k-gguf.zip model

from google.colab import files
files.download("qwen2.5-0.5b-lunarlander-action-100k-gguf.zip")

  adding: model/ (stored 0%)
  adding: model/config.json (deflated 72%)
  adding: model/chat_template.jinja (deflated 75%)
  adding: model/model.safetensors (deflated 21%)
  adding: model/tokenizer.json (deflated 81%)
  adding: model/tokenizer_config.json (deflated 68%)
  adding: model/.cache/ (stored 0%)
  adding: model/.cache/huggingface/ (stored 0%)
  adding: model/.cache/huggingface/download/ (stored 0%)
  adding: model/.cache/huggingface/download/model.safetensors.metadata (deflated 29%)
  adding: model/.cache/huggingface/.gitignore (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
!find /content -name "*.gguf"


/content/model_gguf/qwen2.5-0.5b.Q4_K_M.gguf


In [21]:
!zip -j qwen2.5-0.5b-lunarlander-action-q4_k_m-gguf.zip /content/model_gguf/qwen2.5-0.5b.Q4_K_M.gguf

from google.colab import files
files.download("qwen2.5-0.5b-lunarlander-action-q4_k_m-gguf.zip")

  adding: qwen2.5-0.5b.Q4_K_M.gguf (deflated 3%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in `llama.cpp` or a UI based system like `GPT4All`. You can install GPT4All by going [here](https://gpt4all.io/index.html).

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/drive/1T-YBVfnphoVc8E2E854qF3jdia2Ll2W2?usp=sharing)**

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/u54VK8m8tk) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Zephyr DPO 2x faster [free Colab](https://colab.research.google.com/drive/15vttTpzzVXv_tJwEk-hIcQ0S9FcEWvwP?usp=sharing)
2. Llama 7b 2x faster [free Colab](https://colab.research.google.com/drive/1lBzz5KeZJKXjvivbYvmGarix9Ao6Wxe5?usp=sharing)
3. TinyLlama 4x faster full Alpaca 52K in 1 hour [free Colab](https://colab.research.google.com/drive/1AZghoNBQaMDgWJpi4RbffGM1h6raLUj9?usp=sharing)
4. CodeLlama 34b 2x faster [A100 on Colab](https://colab.research.google.com/drive/1y7A0AxE3y8gdj4AVkl2aZX47Xu3P1wJT?usp=sharing)
5. Mistral 7b [free Kaggle version](https://www.kaggle.com/code/danielhanchen/kaggle-mistral-7b-unsloth-notebook)
6. We also did a [blog](https://huggingface.co/blog/unsloth-trl) with 🤗 HuggingFace, and we're in the TRL [docs](https://huggingface.co/docs/trl/main/en/sft_trainer#accelerate-fine-tuning-2x-using-unsloth)!
7. `ChatML` for ShareGPT datasets, [conversational notebook](https://colab.research.google.com/drive/1Aau3lgPzeZKQ-98h69CCu1UJcvIBLmy2?usp=sharing)
8. Text completions like novel writing [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing)
9. [**NEW**] We make Phi-3 Medium / Mini **2x faster**! See our [Phi-3 Medium notebook](https://colab.research.google.com/drive/1hhdhBa1j_hsymiW9m-WzxQtgqTH_NHqi?usp=sharing)
10. [**NEW**] We make Gemma-2 9b / 27b **2x faster**! See our [Gemma-2 9b notebook](https://colab.research.google.com/drive/1vIrqH5uYDQwsJ4-OO3DErvuv4pBgVwk4?usp=sharing)
11. [**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)
12. [**NEW**] We make Mistral NeMo 12B 2x faster and fit in under 12GB of VRAM! [Mistral NeMo notebook](https://colab.research.google.com/drive/17d3U-CAIwzmbDRqbZ9NnpHxCkmXB6LZ0?usp=sharing)

<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Support our work if you can! Thanks!
</div>